MODELLO nn.MODULE IN PYTORCH

Per disegnare il motore pezzo per pezzo, mentre con Keras si montano solo i pezzi.
Ci permette di passare da utilizzatori di strumenti a veri e propri creatori di architettura.

- Ereditarietà e struttura
- Il Costruttore
- Il metodo Forward

nn.Module è la base vera di PyTorch.
Se Sequential di Keras è 'costruisco una pipeline pronta' nn.Module è 'mi costruisco io la rete pezzo per pezzo'.
E' molto più flessibile di Sequential ed è il motivo per cui PyTorch domina nella ricerca AI.

In PyTorch ogni rete neurale è un oggetto Python.
nn.Module è la classe padre da cui derivano:
- layer (Linear, Conv2d, LSTM)
- modelli completi
- blocchi custom
- reti profonde
La vera potenza arriva quando crei modelli personalizzati.
Senza nn.Module dovresti:
- gestire i pesi manualmente
- gestire i gradienti manualmente
- salvare i parametri manualmente
- spostare tutto su GPU manualmente
Con nn.Module ottieni:
- registrazione automatica parametri
- .parameters()
- .to(device)
- .traing()
- .eval()
- serializzazione
- compatibilità optimizer

In PyTorch una rete neurale è una classe Python.
Questa è la differenza filosofica enorme rispetto a Keras.
Scrivi:
class MyModel(nn.Module)
cioè erediti da nn.Module, definisce layer, definisce forward pass.
Ogni modello PyTorch ha quasi sempre:
1. __init__: definisce i layer
2. forward: definisce il flusso dei dati (logica forward)

Con Class MyModel(nn.Model) stai creando una rete neurale personalizzata. nn.Module contiene: - gestione pesi - gradienti - GPU - training/eval mode - salvataggio modello.

super().__init__()
Inizializza la classe padre nn.Module
Obbligatorio praticamente sempre

Definizione layer
self.fc1=nn.Linear(10,64)
Layer full connected con 10 feature e 64 neuroni output
PyTorch crea automaticamente pesi e bias

Ereditarietà da nn.Module
perchè dobbiamo ereditare da una classe già esistente?
In PyTorch, a differenza dell'approccio sequenziale di Keras, la costruzione di una rete neurale segue il paradigma della programmazione a oggetti. Ogni modello è una classe che eredita da nn.Module.
Questo legame di ereditarietà permette alla nostra classe di acquisire funzionalità cruciali, come il tracciamento dei parametri, il passaggio tra CPU e GPU e la gestione dello stato del modello.
In questo modo PyTorch vede la nostra classe e gestisce tutto il lavoro pesante che sta dietro le quinte.

L'anatomia della classe Modello
I vantaggi dell'approccio Object Oriented
- Incapsulamento: i layer e la logica di calcolo risiedono all'interno di un unico oggetto coerente e riutilizzabile. Hai una scatola nera, fuori è pulita e semplice, dentro contiene ingranaggi complessi
- Registrazione parametri: ogni oggetto nn.Parameter o sottomodulo aggiunto alla classe viene automaticamente rilevato dal framework
- Modularità: è possibile nidificare moduli dentro altri moduli, creando gerarchie di astrazione estremamente complesse.
- Flessibilità: ereditare da nn.Module garantisce la compatibilità con tutti gli ottimizzatori e le utility di salvataggio del framework (model è contenuto in nn.Module)

Il ruolo della funzione super()
- Inizializzazione della classe base: la chiamata a super().__init__() è obbligatoria per permettere a PyTorch di configurare correttamente le strutture dati interne necessarie al modulo. Senza questa chiamata la nostra classe sarebbe come una macchina senza centralina elettrica, i pezzi ci sono ma non comunicano con il sistema. 
- Stato del modulo: PyTorch distingue tra parametri addestrabili e buffer (valori costanti o statistiche), gestendo entrambi attravereso l'interfaccia della classe.
- Passaggio Device: grazie all'ereditarietà invocare .to(device) sulla classe sposta ricorsivamente tutti i layer interni sul processore desiderato.

Tracciamento dei Gradienti
La connessione con Autograd
Quando definiamo una classe come nn.Module, PyTorch predispone il grafo computazionale per monitorare le operazioni sui parametri durante l'esecuzione.
Questo meccanismo assicura che, dopo ogni calcolo, sia possibile risalire lungo la catena delle operazioni per determinare l'impatto di ogni singolo peso sull'errore finale.

Il Costruttore: Definire i Layer
Configurare lo stato interno
Il metodo __init__ è l'unico in cui definiamo cosa compone la nostra rete.
Qui istanziamo i layer lineari, convoluzionali o di attivazione come attributi della classe.
In questa fase non stiamo ancora eseguendo calcoli, ma stiamo preparando la memoria e inizializzando i pesi che verranno poi ottimizzati durante l'addestramento.

Dichiarazione delle Componenti
- nn.Linear: definisce un layer fully connected specificando le dimensioni di input e output
- Activation Layers: le funzioni di attivazione come ReLu o Sigmoid possono essere istanziate come oggetti nel costruttore
- Dimensionamento: è fondamentale che l'output di un layer corrisponda esattamente all'input del layer successivo
- Inizializzazione Custom: nel costruttore è possibile sovrascrivere l'inizializzazione predefinita dei pesi per migliorare la convergenza.

Organizzazoine iterna
- nn.Sequential nel costruttore: è possibile raggruppare blocchi di layer ripetitivi dentro nn.Sequential per mantenere il codice del costruttore pulito e leggibile
- Parametri Statici: valori che non cambiano durante il training, come le dimensioni delle immagini o le costanti di normalizzazione, vengono salvati come attributi semplici.
- Istanze uniche: ogni istanza di layer possiede propri pesi; riutilizzare lo stesso attributo in più punti del forward pass significa condividere gli stessi parametri.

L'importanze dei Pesi
l'importanza del punto di partenza.
Se partiamo dal punto sbagliato potremmo non trovare mai la valle, punto di arrivo.
PyTorch inizializza i layer lineari usando la distribuzione Kaiming o Xavier di default. Questo evita che i gradienti diventino troppo piccoli o esplodano all'inizio del training.
Comprendere come i parametri vengano calcolati nel costruttore è essenziale per diagnosticare modelli che non riescono a iniziare l'apprendimento correttamente.

Il medoto Forward
delinea il cammino dei dati.
Il metodo Forward è il cuore operativo della classe. Qui definiamo la sequenza logica con cui il tensore di input viene trasformato dai layer definiti nel costruttore.
PyTorch chiama automaticamente questo metodo quando invochiamo il modello come una funzione, gestendo internamente le registrazioni dei nodi del grafo.

Esecuzione del calcolo
Flusso tensoriale e logica dinamica
- Passaggio dati: l'input attraversa i layer in ordine, subendo trasformazioni lineari e non lineari
- Logica Dinamica: a differenza di altri framework, è possibile inserire if o for basati sui dati direttamente nel forward pass. E' possibile cambiare la ricetta mentre stiamo già cucinando. Questa libertà totale è il motivo per cui PyTorch domina la ricerca scientifica, la strada la costruisci mentre corri.
- Functional API: operazioni prive di stato, come ReLu, possono essere invocate tramite torch.nn.functional per snellire il codice.
- Output: l'ultimo valore restituito dal metodo forward, rappresenta la previsione finale del modello.

Debugging e Ispezione
Per capire dove si trova l'errore
- Print Debugging: Poiche il forward pass è puro codice Python, è possibile inserire istruzioni 'print' per ispezionare le forme (shape) dei tensori durante l'esecuzione. E' come poter manenere una telecamera per verificare dove si è fermato.
- Reshaping al volo: spesso i dati devono essere 'appiattiti' (flattening) tra un layer e l'altro; questo avviene tipicamente all'interno del motodo forward
- Skip Connections: la struttura flessibile del forward permete di sommare l'input originale all'output di un layer, facilitando la creazione di reti residuali.
Tutta questa flessibilità nasce dal concetto di grafo dinamico

Grafo Dinamico vs Esecuzione
Il concetto di definie-by-Run
Immagina di costruire un ponte mentre ci cammini sopra, finita la corsa il grafo viene distrutto e ricostruito per il grafo successivo.
In PyTorch il grafo viene costruito ogni volta che il metodo forward viene eseguito.
Questo permette di gestire input di dimensioni variabili con estrema facilità
z=W*X+b
Questa natura dinamica rendo lo sviluppo molto più simile alla programmazione standard, facilitando il compito dello sviluppatore nel mappare la teorica sulla pratica.

In [2]:
import torch
import torch.nn as nn # il modulo fondamentale per i layer
import torch.nn.functional as F #per funzioni di attivazione senza parametri

#1. DEFINIZIONE DELLA CLASSE
#Ereditiamo da nn.Module per registrare automaticamente i parametri
class MioClassificatore(nn.Module):
    def __init__(self,input_size, hidden_size, num_classes):
        #Chiamata obbligatoria al costruttore della classe base
        super(MioClassificatore,self).__init__()

        #2. DEFINIZIONE DEI LAYER (COSTRUTTORE)
        #Primo layer lineare: trasforma input_size -> hidden_size
        self.fc1=nn.Linear(input_size,hidden_size)
        #Secondo layer lineare: trasforma hidden_size -> num_classes (output)
        self.fc2=nn.Linear(hidden_size,num_classes)

        #Nota: i pesi e i bias vengono inizializzati automaticamente qui
    
    #3.DEFINIZIONE DEL FLUSSO (FORWARD PASS)
    def forward(self,x):
        #Passaggio nel primo layer
        x=self.fc1(x)
        #Applicazione della non-linearità (ReLu)
        #Usiamo F.relu perchè non ha pesi da addestrare
        x=F.relu(x)
        #Passaggio finale per ottenere i logit di output
        x=self.fc2(x)

        return x

#--- TEST DEL MODELLO ---
#Ipotiziamo: 10 features in ingresso, 20 neuroni nascosti, 3 classi in uscita
input_dim=10
hidden_dim=20
output_dim=3

model=MioClassificatore(input_dim, hidden_dim, output_dim)

#Creiamo un batch di dati finti (5 campioni, 10 feature ciascuno)
dummy_input=torch.randn(5,input_dim)

#Eseguiamo il forward pass chiamando l'oggetto direttamente
output=model(dummy_input)

print(f"Architettura del modello:\n{model}")
print(f"\nShape dell'output: {output.shape}") #dovrebbe essere 5,3
print(f"I gradienti sono attivi? {output.requires_grad}")

Architettura del modello:
MioClassificatore(
  (fc1): Linear(in_features=10, out_features=20, bias=True)
  (fc2): Linear(in_features=20, out_features=3, bias=True)
)

Shape dell'output: torch.Size([5, 3])
I gradienti sono attivi? True
